# Q2 공시 기간·유형 분석 — 이정수 담당

Issue #7. 실제 Open DART SQLite 공시를 Pandas로 집계하고 독립 SQL 집계와 대조한다. 정정 공시는 별도 접수로 포함한다.

In [1]:
from pathlib import Path
from contextlib import closing
import sqlite3, sys
import pandas as pd
root = Path.cwd()
if not (root / 'src').exists(): root = root.parent
sys.path.insert(0, str(root))
from src.disclosure_pandas_leejeongsu import analyze
from src.validation_leejeongsu import compare_results
database = root / 'data' / 'processed' / 'dart.sqlite'
assert database.is_file(), f'로컬 DART DB가 필요합니다: {database}'
print('Pandas:', pd.__version__)

C:\Users\USER\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Pandas: 3.0.4


## 원본과 독립 SQL 집계

기간은 접수일의 연월, 유형은 보고서명 양쪽 공백을 제거한 값이다. 접수번호 중복과 잘못된 날짜는 분석 전에 오류로 중단한다.

In [2]:
query = '''
SELECT corp_code,
       substr(rcept_dt,1,4)||'-'||substr(rcept_dt,5,2) AS period,
       trim(report_nm) AS report_type,
       COUNT(*) AS count,
       SUM(CASE WHEN trim(report_nm) LIKE '[기재정정]%'
                  OR trim(report_nm) LIKE '[첨부추가]%'
                THEN 1 ELSE 0 END) AS correction_count
FROM disclosures
GROUP BY corp_code, period, report_type
'''
with closing(sqlite3.connect(database)) as db:
    raw = pd.read_sql_query('SELECT * FROM disclosures', db)
    sql_result = pd.read_sql_query(query, db)
by_type, monthly = analyze(raw)
print('공시 원본:', len(raw), '기간·유형 그룹:', len(by_type),
      '월별 그룹:', len(monthly), '정정·첨부 접수:', int(by_type.correction_count.sum()))

공시 원본: 438 기간·유형 그룹: 197 월별 그룹: 24 정정·첨부 접수: 30


## SQL과 Pandas 교차검증

In [3]:
checks = compare_results(sql_result, by_type,
                         ['corp_code', 'period', 'report_type'],
                         ['count', 'correction_count'])
assert all(item['status'] == 'PASS' for item in checks), checks
print('SQL-Pandas 일치:', len(by_type), '그룹 / CHECK:',
      sum(item['status'] != 'PASS' for item in checks))

SQL-Pandas 일치: 197 그룹 / CHECK: 0


## 전년 같은 달과 비교한 확인 후보

기본 규칙은 증감률 절댓값 50% 이상 **그리고** 건수 차이 절댓값 10건 이상이다. 이전 같은 달이 없으면 증감률을 계산하지 않는다. 후보는 추가 확인할 지점이며 기업 위험 판정이 아니다.

In [4]:
display(monthly.loc[monthly.period.str.startswith('2024'),
                    ['period', 'count', 'previous_count', 'change_count',
                     'change_pct', 'correction_count', 'candidate']])
_, monthly_100 = analyze(raw, threshold_pct=100, min_change=10)
print('후보 월 수 (50%/10건):', int(monthly.candidate.sum()),
      '(100%/10건):', int(monthly_100.candidate.sum()))
display(by_type.loc[by_type.period.str.startswith('2024')]
        .sort_values('count', ascending=False).head(10))

,period,count,previous_count,change_count,change_pct,correction_count,candidate
12,2024-01,15,12.0,3.0,25.000000,2,False
13,2024-02,17,20.0,-3.0,-15.000000,1,False
14,2024-03,23,6.0,17.0,283.333333,1,True
15,2024-04,16,12.0,4.0,33.333333,3,False
16,2024-05,13,11.0,2.0,18.181818,1,False
17,2024-06,30,5.0,25.0,500.000000,1,True
18,2024-07,13,13.0,0.0,0.000000,1,False
19,2024-08,14,13.0,1.0,7.692308,1,False
20,2024-09,31,4.0,27.0,675.000000,1,True
21,2024-10,37,15.0,22.0,146.666667,5,True


후보 월 수 (50%/10건): 6 (100%/10건): 5


,corp_code,period,report_type,count,correction_count
194,00126380,2024-12,임원ㆍ주요주주특정증권등소유상황보고서,39,0
143,00126380,2024-06,임원ㆍ주요주주특정증권등소유상황보고서,26,0
164,00126380,2024-09,임원ㆍ주요주주특정증권등소유상황보고서,26,0
172,00126380,2024-10,임원ㆍ주요주주특정증권등소유상황보고서,20,0
187,00126380,2024-11,임원ㆍ주요주주특정증권등소유상황보고서,15,0
117,00126380,2024-03,임원ㆍ주요주주특정증권등소유상황보고서,11,0
158,00126380,2024-08,임원ㆍ주요주주특정증권등소유상황보고서,5,0
107,00126380,2024-02,임원ㆍ주요주주특정증권등소유상황보고서,4,0
168,00126380,2024-10,[기재정정]주식등의대량보유상황보고서(일반),4,4
105,00126380,2024-02,동일인등출자계열회사와의상품ㆍ용역거래변경,3,0


## 해석 범위

같은 원본의 SQL/Pandas 계산 일치만 확인했다. 정정 공시도 별도 접수로 세므로 공시 사건의 고유 건수와 다르다. 2023·2024 1개 기업의 2개 연도만 있어 계절성·장기 추세나 이상 원인을 확정할 수 없다. 이영 담당 공시 SQL을 `dev`에 합친 뒤 그 쿼리와 최종 재검증한다.